In [ ]:
import numpy as np
import CI
import CI_physicist

from electron_integrals import *

In [ ]:
# Number of orbitals (without spin)
num_orbitals = 5
# Number of electrons
num_electrons = 3
#Include spin?
include_spin = True

num_spin_orbitals = (1+int(include_spin))*num_orbitals

Calculate electron integrals

In [ ]:
x_max = 10
num_points = 1000

x = np.linspace(-x_max,x_max,num_points)

pot = GaussianWell(w=100, a=1, center=0)
#pot = HOPotential()

spf, h = get_spf_and_diag_h(num_orbitals, x, pot)

if include_spin:
    #Add spin
    h = np.kron(h, np.eye(2,2))

g = coulomb_interaction_matrix_elements(spf, spf, x, x, kappa = 1, a=0.01)

if include_spin:
    #Add spin
    g = np.kron(g, np.einsum("pr,qs->pqrs",np.eye(2,2), np.eye(2,2)))

g_chemist = g.transpose(0,2,1,3)

In [ ]:
H_chemist = CI.AddressHamiltonian(num_spin_orbitals, num_electrons, h, g_chemist).get_hamiltonian()
E_chemist, _ = np.linalg.eigh(H_chemist)
print(E_chemist)

H_physicist = CI_physicist.AddressHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_physicist, _ = np.linalg.eigh(H_physicist)
print(E_physicist)

H_SC_physicist = CI_physicist.SlaterCondonHamiltonian(num_spin_orbitals, num_electrons, h, g).get_hamiltonian()
E_SC_physicist, _ = np.linalg.eigh(H_SC_physicist)
print(E_SC_physicist)

np.testing.assert_allclose(E_chemist, E_physicist)
np.testing.assert_allclose(E_chemist, E_SC_physicist)